In [ ]:
"""
05_modeling.py
---------------
Phases 5-7 combined: train/test split, model training
(Linear Regression + XGBoost), and evaluation.

Reads:
    data/processed/customer_features.csv

Writes:
    data/processed/model_comparison.csv   (RMSE/R2 for both models)
    data/processed/feature_importance.csv (from XGBoost)
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

# ----------------------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------------------
df = pd.read_csv("data/processed/customer_features.csv")
print(f"Loaded {len(df)} customers")

# ----------------------------------------------------------------
# 2. ENCODE CATEGORICAL FEATURES
#    drop_first=True avoids the "dummy variable trap" (perfectly
#    collinear columns), which matters for Linear Regression
#    specifically - XGBoost wouldn't care either way.
# ----------------------------------------------------------------
categorical_cols = ["acquisition_channel", "age_band", "region"]
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

feature_cols = [c for c in df_encoded.columns if c not in ["customer_id", "target_holdout_spend"]]
X = df_encoded[feature_cols]
y = df_encoded["target_holdout_spend"]

print(f"Feature columns used: {feature_cols}\n")

# ----------------------------------------------------------------
# 3. TRAIN / TEST SPLIT (by customer, not by transaction -
#    each row here already IS one customer, so this is correct)
# ----------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train customers: {len(X_train)}")
print(f"Test customers: {len(X_test)}\n")

# ----------------------------------------------------------------
# 4. TRAIN MODEL 1: LINEAR REGRESSION (interpretable baseline)
# ----------------------------------------------------------------
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2 = r2_score(y_test, lr_preds)

# ----------------------------------------------------------------
# 5. TRAIN MODEL 2: XGBOOST (captures non-linear patterns)
# ----------------------------------------------------------------
xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_r2 = r2_score(y_test, xgb_preds)

# ----------------------------------------------------------------
# 6. COMPARE THE TWO MODELS
# ----------------------------------------------------------------
comparison = pd.DataFrame({
    "model": ["Linear Regression", "XGBoost"],
    "RMSE": [round(lr_rmse, 2), round(xgb_rmse, 2)],
    "R2": [round(lr_r2, 4), round(xgb_r2, 4)],
})
print("Model comparison on test set (lower RMSE is better, higher R2 is better):")
print(comparison.to_string(index=False))

winner = "XGBoost" if xgb_r2 > lr_r2 else "Linear Regression"
print(f"\nBetter model on this test set: {winner}")

comparison.to_csv("data/processed/model_comparison.csv", index=False)

# ----------------------------------------------------------------
# 7. FEATURE IMPORTANCE (from XGBoost) - which behaviors actually
#    drive predicted future value
# ----------------------------------------------------------------
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": xgb_model.feature_importances_,
}).sort_values("importance", ascending=False)

print("\nXGBoost feature importance (which factors matter most):")
print(importance.to_string(index=False))

importance.to_csv("data/processed/feature_importance.csv", index=False)

# ----------------------------------------------------------------
# 8. LINEAR REGRESSION COEFFICIENTS (for interpretability comparison)
# ----------------------------------------------------------------
coefficients = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": lr_model.coef_,
}).sort_values("coefficient", ascending=False)

print("\nLinear Regression coefficients (direction and rough size of each effect):")
print(coefficients.to_string(index=False))

print("\nSaved: data/processed/model_comparison.csv")
print("Saved: data/processed/feature_importance.csv")